# Notebook 3: Register Model in Snowflake Model Registry

This notebook:
1. Loads the locally-trained GradientBoostingClassifier from artifacts
2. Registers it in the Snowflake Model Registry
3. Verifies the model works by running inference on test data

**Prerequisites**: Run `01_local_training.ipynb` and `02_snowflake_setup.ipynb` first.

In [ ]:
import json
import joblib
import pandas as pd
import numpy as np
from snowflake.snowpark import Session
from snowflake.ml.registry import Registry
from snowflake.ml.model import task as ml_task

print("Libraries loaded.")

## 1. Connect to Snowflake

In [ ]:
session = Session.builder.config("connection_name", "DEMO").create()
session.sql("USE ROLE ACCOUNTADMIN").collect()
session.sql("USE DATABASE HEALTHCARE_ML").collect()
session.sql("USE SCHEMA MODEL_REGISTRY").collect()
session.sql("USE WAREHOUSE HEALTHCARE_ML_WH").collect()

print(f"Connected: {session.get_current_account()}")
print(f"Database: {session.get_current_database()}")
print(f"Schema: {session.get_current_schema()}")

## 2. Load Local Artifacts

In [ ]:
# Load the trained model
model = joblib.load('artifacts/readmission_model.joblib')
print(f"Model type: {type(model).__name__}")
print(f"Model params: n_estimators={model.n_estimators}, max_depth={model.max_depth}")

# Load metadata
with open('artifacts/model_metadata.json', 'r') as f:
    metadata = json.load(f)

FEATURE_COLUMNS = metadata['feature_columns']
print(f"Features: {len(FEATURE_COLUMNS)} columns")
print(f"Training metrics: ROC AUC={metadata['model_metrics']['roc_auc']}, AP={metadata['model_metrics']['average_precision']}")

In [ ]:
# Load test data for sample_input_data and verification
test_df = pd.read_csv('artifacts/test_data.csv')
print(f"Test data: {test_df.shape[0]} rows, {test_df.shape[1]} columns")

# Prepare sample input (features only, no target)
sample_input = test_df[FEATURE_COLUMNS].head(100)
print(f"Sample input shape: {sample_input.shape}")
print(f"Sample columns: {list(sample_input.columns[:5])}...")

## 3. Register Model in Snowflake Model Registry

The Model Registry stores the model artifacts, dependencies, and metadata in Snowflake.
This makes the model available for:
- Warehouse inference via `mv.run()`
- Distributed batch inference via `mv.run_batch()` on SPCS
- Version management and governance

In [ ]:
# Initialize the registry
registry = Registry(
    session=session,
    database_name="HEALTHCARE_ML",
    schema_name="MODEL_REGISTRY"
)

print(f"Registry initialized in HEALTHCARE_ML.MODEL_REGISTRY")
print(f"Existing models: {len(registry.models())}")

In [ ]:
# Log the model to the registry
mv = registry.log_model(
    model=model,
    model_name="READMISSION_PREDICTOR",
    version_name="V1",
    sample_input_data=sample_input,
    conda_dependencies=["scikit-learn"],
    comment="30-day hospital readmission prediction model. GradientBoostingClassifier trained on 8617 samples with 33 clinical features.",
    metrics={
        "roc_auc": metadata['model_metrics']['roc_auc'],
        "average_precision": metadata['model_metrics']['average_precision'],
        "training_samples": metadata['training_samples'],
        "test_samples": metadata['test_samples'],
        "n_features": len(FEATURE_COLUMNS)
    },
    task=ml_task.Task.TABULAR_BINARY_CLASSIFICATION
)

print(f"Model registered successfully!")
print(f"Model: READMISSION_PREDICTOR, Version: V1")

## 4. Verify Model in Registry

In [ ]:
# List models in registry
models = registry.models()
print(f"Models in registry: {len(models)}")
for m in models:
    print(f"  - {m.name}")
    for v in m.versions():
        print(f"    Version: {v.version_name}")
        print(f"    Comment: {v.comment}")
        print(f"    Metrics: {v.get_metric()}")

## 5. Test Inference via Warehouse (mv.run)

This runs the model on the Snowflake warehouse - suitable for small-to-medium datasets.
For large-scale batch inference, we'll use `run_batch()` in Notebook 4.

In [ ]:
# Retrieve the model version
mv = registry.get_model("READMISSION_PREDICTOR").version("V1")

# Show available functions
print("Available model functions:")
for func in mv.show_functions():
    print(f"  - {func}")

In [ ]:
# Run predict on test samples using the warehouse
test_features = test_df[FEATURE_COLUMNS].head(20)
predictions = mv.run(test_features, function_name="predict")

print("Warehouse Inference Results (first 20 patients):")
print(predictions.head(20).to_string())

In [ ]:
# Run predict_proba for probability scores
probabilities = mv.run(test_features, function_name="predict_proba")

print("Probability Scores (first 10):")
print(probabilities.head(10).to_string())

In [ ]:
# Compare with local predictions to verify consistency
local_preds = model.predict(test_features.values)

# Get the prediction column from Snowflake results
sf_pred_col = [c for c in predictions.columns if 'predict' in c.lower() or 'output' in c.lower()]
if sf_pred_col:
    sf_preds = predictions[sf_pred_col[0]].values
    match_rate = (local_preds == sf_preds).mean()
    print(f"Local vs Snowflake prediction match rate: {match_rate:.1%}")
else:
    print(f"Prediction columns available: {list(predictions.columns)}")
    print("Local predictions:", local_preds[:10])

## 6. Verify via SQL

Models registered in the registry are also accessible via SQL.

In [ ]:
# Check model exists in SQL
sql_models = session.sql("SHOW MODELS IN SCHEMA HEALTHCARE_ML.MODEL_REGISTRY").collect()
print(f"Models found via SQL: {len(sql_models)}")
for m in sql_models:
    print(f"  - {m['name']}")

## 7. Verification Summary

In [ ]:
print("="*60)
print("MODEL REGISTRY VERIFICATION")
print("="*60)
print(f"\nModel: READMISSION_PREDICTOR")
print(f"Version: V1")
print(f"Registry: HEALTHCARE_ML.MODEL_REGISTRY")
print(f"Type: GradientBoostingClassifier (scikit-learn)")
print(f"Features: {len(FEATURE_COLUMNS)}")
print(f"Functions: predict, predict_proba")
print(f"Warehouse inference: VERIFIED")
print(f"\nReady for:")
print(f"  - Notebook 04: Batch inference with run_batch() on SPCS")
print(f"  - Notebook 05: Real-time inference with Online Feature Store")

session.close()
print("\nSession closed.")